# Stability Criteria: Input Parameters

Analyses the **input parameters** required by the Roch skier and WEAC skier stability criteria
on ECTP slabs: which calculation pathways are available, how many slabs have all required inputs
(coverage), and the relative uncertainty of each input parameter by method.

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Find Parameterizations](#2-find-parameterizations)
3. [Roch Skier: Input Parameter Analysis](#3-roch-skier-input-parameter-analysis)
4. [WEAC Skier: Input Parameter Analysis](#4-weac-skier-input-parameter-analysis)
5. [Input Coverage Comparison](#5-input-coverage-comparison)
6. [Save Results](#6-save-results)

| Criterion | Target | Required layer inputs | Pathways |
|-----------|--------|-----------------------|----------|
| Roch skier | S_sk = (τ_c − τ) / τ_sk | density | 4 |
| WEAC skier | g_delta = Γ/Γ_c | density + E-mod + ν + G | 32 |

τ_c and weak-layer fracture parameters are constants (weissgraeber_rosendahl) and always available.

Results are saved to parquet for use by `stability_criteria_outputs.ipynb`.

## 1. Setup & Data Loading

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from snowpyt_mechparams.snowpilot import parse_caaml_directory
from snowpyt_mechparams.models import Pit
from notebook_utils import (
    load_pits, create_ectp_slabs,
    nominal, count_ok, extract_param_stats,
    DENSITY_COLORS, rgba,
)
from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

In [2]:
# Helper functions and DENSITY_COLORS/rgba imported from notebook_utils
# (definitions retained in notebook_utils.py for reuse across notebooks)


In [3]:
pits = load_pits()
print(f'Loaded {len(pits):,} snow pits ({sum(len(p.layers) for p in pits):,} layers)')

ectp_slabs = create_ectp_slabs(pits)

total_slabs = len(ectp_slabs)
print(f'Created {total_slabs:,} ECTP slabs')


Loaded 50,278 snow pits (371,429 layers)
Created 14,776 ECTP slabs


## 2. Find Parameterizations

In [4]:
engine = ExecutionEngine(graph)

roch_paths = find_parameterizations(graph, graph.get_node('s_sk'))
weac_paths = find_parameterizations(graph, graph.get_node('g_delta'))

print(f'Pathways to s_sk   (Roch skier): {len(roch_paths)}')
print(f'Pathways to g_delta (WEAC skier): {len(weac_paths)}')
print()
print('Roch pathways:')
for i, p in enumerate(roch_paths, 1):
    print(f'  {i}:', p)
    print()

Pathways to s_sk   (Roch skier): 4
Pathways to g_delta (WEAC skier): 32

Roch pathways:
  1: branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_roch_inputs
branch 2: snow_pit -- weissgraeber_rosendahl --> tau_c -- data_flow --> merge_roch_inputs
merge branch 1, branch 2: merge_roch_inputs -- roch_skier --> s_sk

  2: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
branch 3: snow_pit -- weissgraeber_rosendahl --> tau_c -- data_flow --> merge_roch_inputs
merge branch 1, branch 2: merge_hand_hardness_grain_form -- geldsetzer --> density
merge branch 1, branch 2, branch 3: merge_roch_inputs -- roch_skier --> s_sk

  3: branch 1: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 2: snow_pit -- data_flow --> measured_grain_form -

## 3. Roch Skier: Input Parameter Analysis

S_sk = (τ_c − τ) / τ_sk requires:
- **density** for every slab layer — 4 methods → 4 pathways
- **τ_c** (constant, weissgraeber_rosendahl — always available)

Coverage is the fraction of ECTP slabs for which **all layers** have a successful density calculation.

In [5]:
config_roch = ExecutionConfig(include_method_uncertainty=False)

roch_slab_records = []   # per (slab, pathway) — for coverage + output

for slab in tqdm(ectp_slabs, desc='Roch s_sk', unit='slab'):
    results  = engine.execute_all(slab, 's_sk', config=config_roch)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        dm = pw.methods_used.get('density', 'data_flow')

        ok_density = count_ok(pw.computation_trace, 'density', n_layers)

        # ── Per-parameter nominal value and relative uncertainty ───────────
        d_nom_avg, d_rel_unc = extract_param_stats(pw.computation_trace, 'density')

        # ── s_sk output ────────────────────────────────────────────────────
        roch_res = pw.slab.roch_skier_result
        s_sk_val = np.nan
        if roch_res is not None and roch_res.stability_index is not None:
            s_sk_val = nominal(roch_res.stability_index)
        s_sk_ok = not np.isnan(s_sk_val)

        roch_slab_records.append({
            'slab_id':          slab.slab_id,
            'density_method':   dm,
            'slab_density_ok':  ok_density,   # kept for downstream compatibility
            'all_inputs_ok':    ok_density,   # τ_c always available → density = all inputs
            's_sk':             s_sk_val,
            's_sk_ok':          s_sk_ok,
            'unstable_roch':    (s_sk_val < 1.0) if s_sk_ok else None,
            # per-parameter stats
            'density_nom_avg':  d_nom_avg,
            'density_rel_unc':  d_rel_unc,
        })

roch_slab_df = pd.DataFrame(roch_slab_records)
print(f'Roch slab records:  {len(roch_slab_df):,}')

Roch s_sk: 100%|██████████| 14776/14776 [00:04<00:00, 3409.62slab/s]

Roch slab records:  59,104


In [6]:
roch_cov = (
    roch_slab_df
    .groupby('density_method')
    .agg(
        n_density_ok    = ('slab_density_ok',  'sum'),
        n_all_inputs    = ('all_inputs_ok',     'sum'),
        n_s_sk_ok       = ('s_sk_ok',           'sum'),
        density_nom     = ('density_nom_avg',   'mean'),
        density_rel_unc = ('density_rel_unc',   'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)
# pct columns retained for coverage-comparison cell downstream
roch_cov['pct_density'] = roch_cov['n_density_ok'] / total_slabs
roch_cov['pct_s_sk']    = roch_cov['n_s_sk_ok']    / total_slabs

# ── Table 1: per-pathway input parameter statistics ───────────────────────────
t1 = pd.DataFrame({
    'ρ_method':           roch_cov['density_method'],
    'ρ_nominal_avg':      roch_cov['density_nom'].map(lambda x: f'{x:.1f}' if not pd.isna(x) else '—'),
    'ρ_avg_rel_unc':      roch_cov['density_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ρ_success_count':    roch_cov['n_density_ok'].astype(int),
    'ρ_success_rate':     roch_cov['pct_density'].map(lambda x: f'{x:.1%}'),
    'τ_c_method':         'weissgraeber_rosendahl',
    'τ_c_note':           'constant — always available',
    'all_inputs_count':   roch_cov['n_all_inputs'].astype(int),
    'all_inputs_rate':    (roch_cov['n_all_inputs'] / total_slabs).map(lambda x: f'{x:.1%}'),
}).reset_index(drop=True)

print('Table 1: Per-pathway input parameter statistics')
print('  ρ in kg/m³ · success = all layers in slab succeeded · τ_c constant (weissgraeber_rosendahl)')
print()
print(t1.to_string(index=False))

# ── Table 2: pathway coverage summary ─────────────────────────────────────────
print()
print()
print('Table 2: Pathway coverage summary')
print(f"  {'Pathway':<28}  {'All inputs':>22}  {'s_sk':>22}")
print(f"  {'-'*78}")
for _, row in roch_cov.iterrows():
    na = int(row['n_all_inputs'])
    ns = int(row['n_s_sk_ok'])
    print(f"  {row['density_method']:<28}  "
          f"{na:>5} / {total_slabs} ({na / total_slabs:.1%})  "
          f"{ns:>5} / {total_slabs} ({ns / total_slabs:.1%})")
print()
print("  'All inputs': all layers have a successful density calculation (τ_c always available).")
print("  's_sk':       Roch skier criterion returned a valid stability index.")

Table 1: Per-pathway input parameter statistics
  ρ in kg/m³ · success = all layers in slab succeeded · τ_c constant (weissgraeber_rosendahl)

           ρ_method ρ_nominal_avg ρ_avg_rel_unc  ρ_success_count ρ_success_rate             τ_c_method                    τ_c_note  all_inputs_count all_inputs_rate
kim_jamieson_table2         177.2         19.1%             5951          40.3% weissgraeber_rosendahl constant — always available              5951           40.3%
         geldsetzer         162.1         19.4%             4539          30.7% weissgraeber_rosendahl constant — always available              4539           30.7%
kim_jamieson_table5         159.4         21.4%             1145           7.7% weissgraeber_rosendahl constant — always available              1145            7.7%
          data_flow         190.9         10.0%              109           0.7% weissgraeber_rosendahl constant — always available               109            0.7%


Table 2: Pathway coverage summ

## 4. WEAC Skier: Input Parameter Analysis

g_delta = Γ/Γ_c requires four layer-level parameters for **every slab layer**:

| Parameter | Methods | Count |
|-----------|---------|-------|
| density | data_flow, geldsetzer, kim_jamieson_table2, kim_jamieson_table5 | 4 |
| elastic modulus | wautier, kochle, bergfeld, schottner | 4 |
| Poisson's ratio | kochle, srivastava | 2 |
| shear modulus | wautier | 1 |

4 × 4 × 2 × 1 = **32 pathways**. Weak-layer fracture parameters are constants (always available).

**Note:** This cell runs WEAC for every slab with `weac_timeout_seconds=5.0` and collects
per-slab input coverage and g_delta outputs. Expect ~25 minutes runtime.

In [7]:
config_weac = ExecutionConfig(include_method_uncertainty=False, weac_timeout_seconds=5.0)

weac_slab_records = []   # per (slab, pathway) — for coverage + output

for slab in tqdm(ectp_slabs, desc='WEAC g_delta', unit='slab'):
    results  = engine.execute_all(slab, 'g_delta', config=config_weac)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        methods = pw.methods_used
        dm  = methods.get('density',         '?')
        em  = methods.get('elastic_modulus',  '?')
        num = methods.get('poissons_ratio',   '?')

        ok_density = count_ok(pw.computation_trace, 'density',        n_layers)
        ok_emod    = count_ok(pw.computation_trace, 'elastic_modulus', n_layers)
        ok_nu      = count_ok(pw.computation_trace, 'poissons_ratio',  n_layers)
        ok_G       = count_ok(pw.computation_trace, 'shear_modulus',   n_layers)
        all_inputs_ok = ok_density and ok_emod and ok_nu and ok_G

        # ── Per-parameter nominal values and relative uncertainties ────────
        d_nom_avg,  d_rel_unc  = extract_param_stats(pw.computation_trace, 'density')
        e_nom_avg,  e_rel_unc  = extract_param_stats(pw.computation_trace, 'elastic_modulus')
        nu_nom_avg, nu_rel_unc = extract_param_stats(pw.computation_trace, 'poissons_ratio')
        G_nom_avg,  G_rel_unc  = extract_param_stats(pw.computation_trace, 'shear_modulus')

        # ── g_delta output ─────────────────────────────────────────────────
        weac_res = pw.slab.weac_result
        g_val    = np.nan
        if weac_res is not None:
            g_val = float(weac_res.g_delta)
        g_ok = not np.isnan(g_val)

        weac_slab_records.append({
            'slab_id':          slab.slab_id,
            'density_method':   dm,
            'emod_method':      em,
            'nu_method':        num,
            'ok_density':       ok_density,
            'ok_emod':          ok_emod,
            'ok_nu':            ok_nu,
            'ok_G':             ok_G,
            'all_inputs_ok':    all_inputs_ok,
            'g_delta':          g_val,
            'g_delta_ok':       g_ok,
            'unstable_weac':    (g_val >= 1.0) if g_ok else None,
            # per-parameter stats
            'density_nom_avg':  d_nom_avg,
            'density_rel_unc':  d_rel_unc,
            'emod_nom_avg':     e_nom_avg,
            'emod_rel_unc':     e_rel_unc,
            'nu_nom_avg':       nu_nom_avg,
            'nu_rel_unc':       nu_rel_unc,
            'G_nom_avg':        G_nom_avg,
            'G_rel_unc':        G_rel_unc,
        })

weac_slab_df = pd.DataFrame(weac_slab_records)
print(f'WEAC slab records:  {len(weac_slab_df):,}')

WEAC g_delta: 100%|██████████| 14776/14776 [07:06<00:00, 34.63slab/s] 


WEAC slab records:  472,832


In [8]:
weac_cov = (
    weac_slab_df
    .groupby(['density_method', 'emod_method', 'nu_method'])
    .agg(
        # slab-level coverage (all layers must succeed)
        n_density_ok    = ('ok_density',       'sum'),
        n_emod_ok       = ('ok_emod',           'sum'),
        n_nu_ok         = ('ok_nu',             'sum'),
        n_G_ok          = ('ok_G',             'sum'),
        n_all_inputs    = ('all_inputs_ok',     'sum'),
        n_g_delta_ok    = ('g_delta_ok',        'sum'),
        # per-parameter average nominal values
        density_nom     = ('density_nom_avg',   'mean'),
        emod_nom        = ('emod_nom_avg',      'mean'),
        nu_nom          = ('nu_nom_avg',        'mean'),
        G_nom           = ('G_nom_avg',         'mean'),
        # per-parameter average relative uncertainty
        density_rel_unc = ('density_rel_unc',   'mean'),
        emod_rel_unc    = ('emod_rel_unc',      'mean'),
        nu_rel_unc      = ('nu_rel_unc',        'mean'),
        G_rel_unc       = ('G_rel_unc',         'mean'),
    )
    .reset_index()
    .sort_values('n_all_inputs', ascending=False)
)

# ── Table 1: per-pathway parameter stats ──────────────────────────────────────
t1 = pd.DataFrame({
    'ρ_method':           weac_cov['density_method'],
    'ρ_nominal_avg':      weac_cov['density_nom'].map(lambda x: f'{x:.1f}' if not pd.isna(x) else '—'),
    'ρ_avg_rel_unc':      weac_cov['density_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ρ_success_count':    weac_cov['n_density_ok'].astype(int),
    'ρ_success_rate':     (weac_cov['n_density_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'E_method':           weac_cov['emod_method'],
    'E_nominal_avg':      weac_cov['emod_nom'].map(lambda x: f'{x:.0f}' if not pd.isna(x) else '—'),
    'E_avg_rel_unc':      weac_cov['emod_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'E_success_count':    weac_cov['n_emod_ok'].astype(int),
    'E_success_rate':     (weac_cov['n_emod_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'ν_method':           weac_cov['nu_method'],
    'ν_nominal_avg':      weac_cov['nu_nom'].map(lambda x: f'{x:.4f}' if not pd.isna(x) else '—'),
    'ν_avg_rel_unc':      weac_cov['nu_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'ν_success_count':    weac_cov['n_nu_ok'].astype(int),
    'ν_success_rate':     (weac_cov['n_nu_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'G_method':           'wautier',
    'G_nominal_avg':      weac_cov['G_nom'].map(lambda x: f'{x:.0f}' if not pd.isna(x) else '—'),
    'G_avg_rel_unc':      weac_cov['G_rel_unc'].map(lambda x: f'{x:.1%}' if not pd.isna(x) else '—'),
    'G_success_count':    weac_cov['n_G_ok'].astype(int),
    'G_success_rate':     (weac_cov['n_G_ok'] / total_slabs).map(lambda x: f'{x:.1%}'),

    'all_inputs_count':   weac_cov['n_all_inputs'].astype(int),
    'all_inputs_rate':    (weac_cov['n_all_inputs'] / total_slabs).map(lambda x: f'{x:.1%}'),
}).reset_index(drop=True)

print('Table 1: Per-pathway input parameter statistics')
print('  ρ in kg/m³ · E and G in Pa · ν dimensionless · success = all layers in slab succeeded')
print()
print(t1.to_string(index=False))

# ── Table 2: pathway coverage summary ─────────────────────────────────────────
print()
print()
print('Table 2: Pathway coverage summary')
print(f"  {'Pathway':<55}  {'All inputs':>16}  {'g_delta':>16}")
print(f"  {'-'*93}")
for _, row in weac_cov.iterrows():
    pw  = f"{row['density_method']} → {row['emod_method']} → {row['nu_method']}"
    na  = int(row['n_all_inputs'])
    ng  = int(row['n_g_delta_ok'])
    print(f"  {pw:<55}  "
          f"{na:>5} / {total_slabs} ({na / total_slabs:.1%})  "
          f"{ng:>5} / {total_slabs} ({ng / total_slabs:.1%})")
print()
print('  Coverage = % ECTP slabs where all layers have the parameter computed.')

Table 1: Per-pathway input parameter statistics
  ρ in kg/m³ · E and G in Pa · ν dimensionless · success = all layers in slab succeeded

           ρ_method ρ_nominal_avg ρ_avg_rel_unc  ρ_success_count ρ_success_rate  E_method E_nominal_avg E_avg_rel_unc  E_success_count E_success_rate   ν_method ν_nominal_avg ν_avg_rel_unc  ν_success_count ν_success_rate G_method G_nominal_avg G_avg_rel_unc  G_success_count G_success_rate  all_inputs_count all_inputs_rate
kim_jamieson_table2         177.2         19.1%             5951          40.3%   wautier           271         36.2%             2092          14.2%     kochle        0.1537          0.0%              926           6.3%  wautier            98         38.8%             2092          14.2%               737            5.0%
kim_jamieson_table2         177.2         19.1%             5951          40.3% schottner             6         88.7%             2780          18.8%     kochle        0.1537          0.0%              926          

In [9]:
TOP_N = 12
top_pw = weac_cov.head(TOP_N).copy()
top_pw['label'] = (
    top_pw['density_method'] + ' → ' +
    top_pw['emod_method']    + ' → ' +
    top_pw['nu_method']
)

steps        = ['n_density_ok', 'n_emod_ok', 'n_nu_ok', 'n_G_ok', 'n_all_inputs', 'n_g_delta_ok']
step_labels  = ['ρ (density)', 'E (elastic mod.)', 'ν (Poisson)', 'G (shear mod.)', 'All inputs', 'g_delta']
step_colors  = ['#0072B2',     '#009E73',           '#E69F00',     '#CC79A7',        '#56B4E9',    '#D55E00']

fig = go.Figure()
for step, label, color in zip(steps, step_labels, step_colors):
    vals = (top_pw[step] / total_slabs * 100).tolist()
    fig.add_trace(go.Bar(
        name=label,
        x=top_pw['label'].tolist(),
        y=vals,
        marker_color=rgba(color, 0.80),
        text=[f'{v:.1f}%' for v in vals],
        textposition='outside',
        textfont=dict(size=8),
    ))

fig.update_layout(
    title=dict(
        text=(
            f'<b>WEAC Skier — Input Parameter Coverage (Top {TOP_N} Pathways by All-Input Coverage)</b><br>'
            '<sup>Coverage = % ECTP slabs where all layers have the parameter computed</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    barmode='group',
    xaxis=dict(tickangle=-35, tickfont=dict(size=9)),
    yaxis=dict(title='Coverage (%)', gridcolor='rgba(200,200,200,0.4)'),
    plot_bgcolor='white',
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.30),
    width=1100, height=520,
    margin=dict(l=60, r=40, t=90, b=180),
)
fig.show()

## 5. Input Coverage Comparison

Roch skier requires only density; WEAC requires density + E-mod + ν + G.
The left panel shows Roch coverage per density method; the right panel shows WEAC coverage for the top
pathways. Colour encodes the density method, consistent across both panels.

In [10]:
TOP_WEAC = 12

roch_sorted = roch_cov.sort_values('n_all_inputs')
weac_top    = weac_cov.head(TOP_WEAC).copy()
weac_top['label'] = (
    weac_top['density_method'] + ' → ' +
    weac_top['emod_method']    + ' → ' +
    weac_top['nu_method']
)
weac_top = weac_top.sort_values('n_all_inputs')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Roch Skier (density input only, 4 pathways)',
        f'WEAC Skier (all 4 layer inputs, top {TOP_WEAC} pathways)',
    ],
    horizontal_spacing=0.14,
)

for _, row in roch_sorted.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['n_all_inputs'] / total_slabs * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[dm], orientation='h',
            marker_color=rgba(color, 0.80),
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=1,
    )

for _, row in weac_top.iterrows():
    dm    = row['density_method']
    color = DENSITY_COLORS.get(dm, '#888')
    pct   = row['n_all_inputs'] / total_slabs * 100
    fig.add_trace(
        go.Bar(
            x=[pct], y=[row['label']], orientation='h',
            marker_color=rgba(color, 0.55),
            marker_line_color=rgba(color, 0.90),
            marker_line_width=1.5,
            text=[f'{pct:.1f}%'], textposition='outside',
            showlegend=False,
        ),
        row=1, col=2,
    )

fig.update_xaxes(title_text='Coverage (%)', range=[0, 100], gridcolor='rgba(200,200,200,0.4)')
fig.update_yaxes(tickfont=dict(size=9))
fig.update_layout(
    title=dict(
        text=(
            '<b>Input Coverage: Roch Skier vs WEAC Skier</b><br>'
            '<sup>% ECTP slabs with all required layer inputs computed — colour = density method</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    plot_bgcolor='white',
    width=1100, height=480,
    margin=dict(l=60, r=80, t=90, b=60),
)
fig.show()

best_roch = roch_cov['n_all_inputs'].max() / total_slabs * 100
best_weac = weac_cov['n_all_inputs'].max() / total_slabs * 100
print(f'Best Roch pathway coverage:  {best_roch:.1f}%')
print(f'Best WEAC pathway coverage:  {best_weac:.1f}%')
print(f'Coverage gap (best Roch − best WEAC): {best_roch - best_weac:.1f} percentage points')

Best Roch pathway coverage:  40.3%
Best WEAC pathway coverage:  5.0%
Coverage gap (best Roch − best WEAC): 35.3 percentage points


## 6. Save Results

Saves `roch_slab_df` and `weac_slab_df` to parquet files for use by
`stability_criteria_outputs.ipynb`, which loads these directly without re-running
the ~25-minute WEAC execution.

In [11]:
roch_slab_df.to_parquet('roch_slab_results.parquet', index=False, engine='pyarrow')
weac_slab_df.to_parquet('weac_slab_results.parquet', index=False, engine='pyarrow')
Path('total_slabs.txt').write_text(str(total_slabs))

print(f'Saved roch_slab_results.parquet  ({len(roch_slab_df):,} rows)')
print(f'Saved weac_slab_results.parquet  ({len(weac_slab_df):,} rows)')
print(f'Saved total_slabs.txt            ({total_slabs:,})')
print()
print('Run stability_criteria_outputs.ipynb to compare stability criterion outputs.')

Saved roch_slab_results.parquet  (59,104 rows)
Saved weac_slab_results.parquet  (472,832 rows)
Saved total_slabs.txt            (14,776)

Run stability_criteria_outputs.ipynb to compare stability criterion outputs.
